# Regression - Prostate Cancer

In [99]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error

In [100]:
prostate_cancer = pd.read_csv("/Users/maddiemann/Documents/MATH-4025-sp26/Math-4025-Project-Maddie-Mann/Data/prad_tcga_pan_can_atlas_2018_clinical_data.tsv", sep='\t')

In [101]:
prostate_cancer.columns = prostate_cancer.columns.str.replace(' ', '_')
prostate_cancer.columns = prostate_cancer.columns.str.replace(r'[\(\)]', '', regex=True)

In [102]:
non_predictor = ['Study_ID', 'Patient_ID', 'Sample_ID', 'Patient_Weight', 
                'Cancer_Type', 'Cancer_Type_Detailed', 'Sex', 'Number_of_Samples_Per_Patient', 
                'American_Joint_Committee_on_Cancer_Metastasis_Stage_Code', 
                'Primary_Lymph_Node_Presentation_Assessment', 'Neoplasm_Disease_Stage_American_Joint_Committee_on_Cancer_Code', 
                'American_Joint_Committee_on_Cancer_Publication_Version_Type', 
                'Neoplasm_Disease_Stage_American_Joint_Committee_on_Cancer_Code', 'TCGA_PanCanAtlas_Cancer_Type_Acronym',
                'Last_Alive_Less_Initial_Pathologic_Diagnosis_Date_Calculated_Day_Value', 'Disease-specific_Survival_status',
                'Ethnicity_Category', 'Neoplasm_Histologic_Grade', 'Neoadjuvant_Therapy_Type_Administered_Prior_To_Resection_Text', 'ICD-10_Classification', 'International_Classification_of_Diseases_for_Oncology,_Third_Edition_ICD-O-3_Histology_Code',
                'International_Classification_of_Diseases_for_Oncology,_Third_Edition_ICD-O-3_Site_Code', 
                'Informed_consent_verified', 'Oncotree_Code', 'Other_Patient_ID',
                'American_Joint_Committee_on_Cancer_Metastasis_Stage_Code', 'Primary_Lymph_Node_Presentation_Assessment', 
                'Race_Category', 'Number_of_Samples_Per_Patient', 'Sample_Type',
                'Somatic_Status', 'Subtype', 'Tissue_Source_Site', 'Tissue_Prospective_Collection_Indicator',
       'Tissue_Retrospective_Collection_Indicator', 'Tissue_Source_Site_Code', 'Tumor_Disease_Anatomic_Site','Last_Communication_Contact_from_Initial_Pathologic_Diagnosis_Date', 'Birth_from_Initial_Pathologic_Diagnosis_Date',
       'Form_completion_date']

In [103]:
prostate_cancer = prostate_cancer.drop(columns =non_predictor)

In [104]:
prostate_cancer= prostate_cancer.rename(columns={
    'Neoplasm_Disease_Lymph_Node_Stage_American_Joint_Committee_on_Cancer_Code': 'Lymph_Node_Stage',
    'American_Joint_Committee_on_Cancer_Tumor_Stage_Code': 'Tumor_Stage' 
})

In [105]:
# Dropping the other potential target variables
other_target = ['Disease_Free_Months', 'Disease_Free_Status','Progression_Free_Status', 'Months_of_disease-specific_survival', 'Overall_Survival_Months', 'Overall_Survival_Status', 'New_Neoplasm_Event_Post_Initial_Therapy_Indicator', 'Person_Neoplasm_Cancer_Status']
prostate_regression = prostate_cancer.drop(columns =other_target)

In [106]:
prostate_cancer['Overall_Survival_Months'].isna().sum()

np.int64(0)

In [107]:
prostate_regression.columns

Index(['Diagnosis_Age', 'Aneuploidy_Score', 'Buffa_Hypoxia_Score',
       'Fraction_Genome_Altered', 'Genetic_Ancestry_Label',
       'In_PanCan_Pathway_Analysis', 'MSI_MANTIS_Score', 'MSIsensor_Score',
       'Mutation_Count', 'Lymph_Node_Stage', 'Tumor_Stage',
       'Progress_Free_Survival_Months', 'Prior_Diagnosis', 'Radiation_Therapy',
       'Ragnum_Hypoxia_Score', 'Tumor_Break_Load', 'TMB_nonsynonymous',
       'Tumor_Type', 'Winter_Hypoxia_Score'],
      dtype='str')

In [108]:
X = prostate_regression.drop(columns = 'Progress_Free_Survival_Months')
y = prostate_regression['Progress_Free_Survival_Months']

In [109]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    train_size= 0.7,
    random_state=4025,
)

In [110]:
cat_vars = ['Genetic_Ancestry_Label', 'In_PanCan_Pathway_Analysis', 'Lymph_Node_Stage', 'Tumor_Stage', 'Prior_Diagnosis', 'Radiation_Therapy', 'Tumor_Type']
num_vars = ['Diagnosis_Age', 'Aneuploidy_Score', 'Buffa_Hypoxia_Score', 'Fraction_Genome_Altered', 'MSI_MANTIS_Score', 'MSIsensor_Score', 'Mutation_Count', 'Ragnum_Hypoxia_Score', 'Tumor_Break_Load', 'TMB_nonsynonymous', 'Winter_Hypoxia_Score']

In [111]:
cat_pipe = Pipeline([
    ('onehot', OneHotEncoder(handle_unknown='ignore'))

])
num_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median'))
])
preprocessor = ColumnTransformer([
    ('cat', cat_pipe, cat_vars),
    ('num', num_pipe, num_vars),
])

tree_pipe = Pipeline([
    ('preproc', preprocessor),
    ('tree', DecisionTreeRegressor(random_state=4025, max_depth = 10))
])

In [112]:
tree_pipe.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preproc', ...), ('tree', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cat', ...), ('num', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains

In [113]:
y_pred = tree_pipe.predict(X_train)
print(f'The tree train r2 is {r2_score(y_train, y_pred)}')
tree_mse = mean_squared_error(y_train, y_pred)
print(f'The tree train rmse is {np.sqrt(tree_mse)}')

The tree train r2 is 0.712297420354246
The tree train rmse is 13.134142189517693


In [114]:
y_pred_test = tree_pipe.predict(X_test)
print(f'The tree test r2 is {r2_score(y_test, y_pred_test)}')
tree_mse = mean_squared_error(y_test, y_pred_test)
print(f'The tree test rmse is {np.sqrt(tree_mse)}')

The tree test r2 is -0.8920752182113523
The tree test rmse is 35.48201053429235


Pivot away from that 

In [115]:
from sklearn.linear_model import LinearRegression

In [116]:
prostate_cancer_clean = prostate_cancer.dropna(subset=['Fraction_Genome_Altered'])

In [117]:
X_lr = prostate_cancer_clean[['Ragnum_Hypoxia_Score', 'Buffa_Hypoxia_Score', 'Winter_Hypoxia_Score']]
y_lr = prostate_cancer_clean['Fraction_Genome_Altered']

In [118]:
prostate_cancer['Fraction_Genome_Altered'].isna().sum()

np.int64(5)

In [119]:
X_train, X_test, y_train, y_test = train_test_split(
    X_lr, y_lr,
    train_size=0.7,
    random_state=4025
)

In [120]:
hypoxia_vars = ['Buffa_Hypoxia_Score', 'Winter_Hypoxia_Score', 'Ragnum_Hypoxia_Score']
num_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='mean'))
])
preprocessor_lr = ColumnTransformer([
    ('num', num_pipe, hypoxia_vars),
])

lr_pipe = Pipeline([
    ('preproc', preprocessor_lr),
    ('model', LinearRegression())
])
lr_pipe.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preproc', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matri

In [121]:
lr_pred = lr_pipe.predict(X_train)
print(f'The hypoxia train r2 is {r2_score(y_train, lr_pred)}')
lr_mse = mean_squared_error(y_train, lr_pred)
print(f'The hypoxia train rmse is {np.sqrt(lr_mse)}')



The hypoxia train r2 is 0.20153637161003468
The hypoxia train rmse is 0.08902446955964677


In [122]:
prostate_cancer['Fraction_Genome_Altered'].mean()

np.float64(0.09043844580777097)

In [123]:
reg_tree_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('tree', DecisionTreeRegressor(max_depth=4, random_state=42)) 
])


reg_tree_pipe.fit(X_train, y_train)


print(f"Train R^2: {reg_tree_pipe.score(X_train, y_train):.3f}")
print(f"Test R^2: {reg_tree_pipe.score(X_test, y_test):.3f}")

Train R^2: 0.347
Test R^2: 0.094


In [ ]:
X_expanded = prostate_cancer_clean[[
    'Ragnum_Hypoxia_Score', 
    'Buffa_Hypoxia_Score', 
    'MSI_MANTIS_Score', 
    'Mutation_Count',       
    'Diagnosis_Age'         
]]

y = prostate_cancer_clean['Fraction_Genome_Altered']

X_train, X_test, y_train, y_test = train_test_split(
    X_expanded, y,
    train_size=0.7,
    random_state=4025
)

reg_tree_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('tree', DecisionTreeRegressor(max_depth=4, random_state=4025)) 
])


reg_tree_pipe.fit(X_train, y_train)


print(f"Train R^2: {reg_tree_pipe.score(X_train, y_train):.3f}")
print(f"Test R^2: {reg_tree_pipe.score(X_test, y_test):.3f}")

Train R^2: 0.393
Test R^2: 0.241


In [126]:
X_train, X_test, y_train, y_test = train_test_split(
    X_expanded, y,
    train_size=0.7,
    random_state=4025
)

reg_tree_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('tree', DecisionTreeRegressor(max_depth=4, random_state=4025)) 
])


reg_tree_pipe.fit(X_train, y_train)


print(f"Train R^2: {reg_tree_pipe.score(X_train, y_train):.3f}")
print(f"Test R^2: {reg_tree_pipe.score(X_test, y_test):.3f}")

Train R^2: 0.393
Test R^2: 0.241


In [127]:
importances = reg_tree_pipe.named_steps['tree'].feature_importances_


feature_names = X_train.columns 


rf_importance_df = pd.Series(importances, index=feature_names).sort_values(ascending=False)

print(rf_importance_df)

Ragnum_Hypoxia_Score    0.446295
Mutation_Count          0.333811
Buffa_Hypoxia_Score     0.113997
MSI_MANTIS_Score        0.105897
Diagnosis_Age           0.000000
dtype: float64


In [128]:
importances = reg_tree_pipe.named_steps['tree'].feature_importances_
feature_names = X_expanded.columns


importance_df = pd.Series(importances, index=feature_names).sort_values(ascending=False)
print(importance_df)

Ragnum_Hypoxia_Score    0.446295
Mutation_Count          0.333811
Buffa_Hypoxia_Score     0.113997
MSI_MANTIS_Score        0.105897
Diagnosis_Age           0.000000
dtype: float64


In [129]:
prostate_cancer_clean2 = prostate_cancer.dropna(subset=['Mutation_Count'])

In [130]:
X_expanded2 = prostate_cancer_clean2[[
    'Ragnum_Hypoxia_Score', 
    'Buffa_Hypoxia_Score',  
    'MSI_MANTIS_Score',      
    'Diagnosis_Age'        
]]

y = np.log1p(prostate_cancer_clean2['Mutation_Count'])

In [131]:
prostate_cancer['Mutation_Count'].isna().sum()

np.int64(3)

In [132]:
X_train, X_test, y_train, y_test = train_test_split(
    X_expanded2, y,
    train_size=0.7,
    random_state=4025
)

reg_tree_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('tree', DecisionTreeRegressor(max_depth=4, min_samples_leaf = 20, random_state=4025)) 
])


reg_tree_pipe.fit(X_train, y_train)


print(f"Train R^2: {reg_tree_pipe.score(X_train, y_train):.3f}")
print(f"Test R^2: {reg_tree_pipe.score(X_test, y_test):.3f}")

Train R^2: 0.124
Test R^2: 0.052


In [133]:
importances = reg_tree_pipe.named_steps['tree'].feature_importances_
pd.Series(importances, index=X_expanded2.columns).sort_values(ascending=False)

feature_names = reg_tree_pipe.named_steps['tree'].get_feature_names_out()

rf_importance_df = pd.Series(importances, index=feature_names).sort_values(ascending=False)
top_features = rf_importance_df.head(10)

plt.figure(figsize=(8, 5))

plt.barh(top_features.index[::-1], top_features.values[::-1])

plt.xlabel("Feature Importance")
plt.title("Top 10 Regression Random Forest Feature Importances")

plt.tight_layout()
plt.show()

AttributeError: 'DecisionTreeRegressor' object has no attribute 'get_feature_names_out'